# 01 Preprocessing
## 전처리

Raw data audit and loan-level model table build.


## Data files
### 데이터 파일


In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

candidate_raw_dirs = []
if os.environ.get("XENTE_RAW_DIR"):
    candidate_raw_dirs.append(Path(os.environ["XENTE_RAW_DIR"]))
candidate_raw_dirs.extend([
    Path.cwd() / "data" / "raw",
    Path.cwd().parent / "data" / "raw",
    Path("/content/drive/MyDrive/XENTE DATA/xente_extracted"),
])

RAW_DIR = next((path for path in candidate_raw_dirs if path.exists()), candidate_raw_dirs[-1])

candidate_processed_dirs = []
if os.environ.get("XENTE_PROCESSED_DIR"):
    candidate_processed_dirs.append(Path(os.environ["XENTE_PROCESSED_DIR"]))
candidate_processed_dirs.extend([
    Path.cwd().parent / "outputs" / "tables",
    Path.cwd() / "outputs" / "tables",
    Path.cwd().parent / "processed",
    Path.cwd() / "processed",
    Path("/content/drive/MyDrive/XENTE DATA/processed"),
])

PROCESSED_DIR = candidate_processed_dirs[0]
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def find_first_existing(filenames):
    for filename in filenames:
        path = RAW_DIR / filename
        if path.exists():
            return path
    raise FileNotFoundError(f"None of these files exist in {RAW_DIR}: {filenames}")


paths = {
    "train": find_first_existing(["Train.csv", "train.csv", "training.csv", "train_final.csv"]),
    "test": find_first_existing(["Test.csv", "test.csv", "test_final.csv"]),
    "unlinked": find_first_existing(["unlinked_masked_final.csv", "unlinked.csv"]),
}

optional_paths = {
    "variable_definitions": ["VariableDefinitions.csv", "variable_definitions.csv"],
    "sample_submission": ["sample_submission.csv", "SampleSubmission.csv"],
}

for name, filenames in optional_paths.items():
    for filename in filenames:
        path = RAW_DIR / filename
        if path.exists():
            paths[name] = path
            break

datasets = {name: pd.read_csv(path) for name, path in paths.items()}

shape_summary = pd.DataFrame([
    {"dataset": name, "path": str(path), "rows": len(datasets[name]), "columns": datasets[name].shape[1]}
    for name, path in paths.items()
])
display(shape_summary)


,dataset,path,rows,columns
0,train,C:\Users\박동하\Documents\Codex\2026-05-16\files-...,2100,27
1,test,C:\Users\박동하\Documents\Codex\2026-05-16\files-...,905,19
2,unlinked,C:\Users\박동하\Documents\Codex\2026-05-16\files-...,16327,12
3,variable_definitions,C:\Users\박동하\Documents\Codex\2026-05-16\files-...,27,2
4,sample_submission,C:\Users\박동하\Documents\Codex\2026-05-16\files-...,478,2


## Target distribution
### 타깃 분포


In [3]:
train = datasets["train"].copy()
loan_rows = train[train["IsDefaulted"].notna()].copy()

target_summary = (
    loan_rows["IsDefaulted"]
    .astype(int)
    .value_counts(dropna=False)
    .rename_axis("is_defaulted")
    .reset_index(name="row_count")
)
target_summary["share_pct"] = target_summary["row_count"] / target_summary["row_count"].sum() * 100
display(target_summary)


,is_defaulted,row_count,share_pct
0,0,1310,88.037634
1,1,178,11.962366


## Date range
### 날짜 범위


In [4]:
date_columns = ["TransactionStartTime", "IssuedDateLoan", "PaidOnDate", "DueDate"]
date_rows = []

for dataset_name, dataset in datasets.items():
    temp = dataset.copy()
    for column in date_columns:
        if column not in temp.columns:
            continue
        if dataset_name == "unlinked" and column == "TransactionStartTime":
            parsed = pd.to_datetime(temp[column], errors="coerce", format="%d/%m/%y %H:%M:%S")
        else:
            parsed = pd.to_datetime(temp[column], errors="coerce")
        date_rows.append({
            "dataset": dataset_name,
            "column": column,
            "non_missing_count": int(parsed.notna().sum()),
            "min_date": parsed.min(),
            "max_date": parsed.max(),
        })

date_summary = pd.DataFrame(date_rows)
display(date_summary)


,dataset,column,non_missing_count,min_date,max_date
0,train,TransactionStartTime,2100,2018-09-21 12:17:39,2019-03-31 12:16:33
1,train,IssuedDateLoan,1488,2018-10-18 16:11:53,2019-03-31 12:16:32
2,train,PaidOnDate,1488,2018-10-22 09:13:17,2019-07-15 12:51:23
3,train,DueDate,1486,2018-11-17 16:11:04,2019-04-30 12:16:26
4,test,TransactionStartTime,905,2019-03-31 13:33:05,2019-07-17 05:34:45
5,test,IssuedDateLoan,478,2019-03-31 13:33:04,2019-07-15 17:06:20
6,unlinked,TransactionStartTime,16327,2019-01-01 00:12:01,2019-06-30 22:57:49


## ID check
### ID 점검


In [5]:
id_columns = ["CustomerId", "TransactionId", "LoanId", "LoanApplicationId", "PayBackId", "BatchId", "SubscriptionId"]
id_rows = []

for column in id_columns:
    if column not in loan_rows.columns:
        continue
    id_rows.append({
        "column": column,
        "unique_count": loan_rows[column].nunique(dropna=False),
        "missing_count": int(loan_rows[column].isna().sum()),
        "duplicated_rows": int(loan_rows.duplicated(column, keep=False).sum()),
    })

id_summary = pd.DataFrame(id_rows)
display(id_summary)


,column,unique_count,missing_count,duplicated_rows
0,CustomerId,239,0,1426
1,TransactionId,1153,0,490
2,LoanId,1159,0,479
3,LoanApplicationId,1157,5,481
4,PayBackId,1485,0,6
5,BatchId,1126,0,535
6,SubscriptionId,5,0,1488


## Missing values
### 결측치


In [6]:
missing_rows = []
for dataset_name, dataset in datasets.items():
    missing_rate = dataset.isna().mean().sort_values(ascending=False)
    for column, rate in missing_rate.head(20).items():
        missing_rows.append({
            "dataset": dataset_name,
            "column": column,
            "missing_count": int(dataset[column].isna().sum()),
            "missing_rate_pct": rate * 100,
        })

missing_summary = pd.DataFrame(missing_rows)
display(missing_summary)


,dataset,column,missing_count,missing_rate_pct
0,train,LoanApplicationId,617,29.380952
1,train,DueDate,614,29.238095
2,train,ThirdPartyId,614,29.238095
3,train,AmountLoan,612,29.142857
4,train,Currency,612,29.142857
5,train,LoanId,612,29.142857
6,train,PaidOnDate,612,29.142857
7,train,IssuedDateLoan,612,29.142857
8,train,IsThirdPartyConfirmed,612,29.142857
9,train,PayBackId,612,29.142857


## Loan rows and transaction pool
### 대출 행과 거래 풀


In [8]:
def find_first_existing(filenames):
    for filename in filenames:
        path = RAW_DIR / filename
        if path.exists():
            return path
    raise FileNotFoundError(f"None of these files exist in {RAW_DIR}: {filenames}")


train_path = find_first_existing(["Train.csv", "train.csv", "training.csv", "train_final.csv"])
unlinked_path = find_first_existing(["unlinked_masked_final.csv", "unlinked.csv"])

train = pd.read_csv(train_path)
unlinked = pd.read_csv(unlinked_path)

for column in ["TransactionStartTime", "IssuedDateLoan", "PaidOnDate", "DueDate"]:
    if column in train.columns:
        train[column] = pd.to_datetime(train[column], errors="coerce")
if "TransactionStartTime" in unlinked.columns:
    unlinked["TransactionStartTime"] = pd.to_datetime(
        unlinked["TransactionStartTime"],
        errors="coerce",
        format="%d/%m/%y %H:%M:%S",
    )

loan_rows = train[train["IsDefaulted"].notna()].copy()
loan_rows["IsDefaulted"] = loan_rows["IsDefaulted"].astype(int)

display(pd.DataFrame([
    {"table": "loan_rows", "rows": len(loan_rows), "columns": loan_rows.shape[1]},
    {"table": "unlinked", "rows": len(unlinked), "columns": unlinked.shape[1]},
]))


,table,rows,columns
0,loan_rows,1488,27
1,unlinked,16327,12


In [9]:
loan_base = (
    loan_rows
    .sort_values(["TransactionId", "TransactionStartTime"])
    .groupby("TransactionId", as_index=False)
    .first()
)

safe_columns = [
    "TransactionId", "CustomerId", "BatchId", "SubscriptionId", "CurrencyCode",
    "CountryCode", "ProviderId", "ProductId", "ProductCategory", "ChannelId",
    "Amount", "Value", "PricingStrategy", "IssuedDateLoan", "IsDefaulted",
    "LoanId", "LoanApplicationId", "PayBackId", "InvestorId",
]
safe_columns = [column for column in safe_columns if column in loan_base.columns]
loan_base = loan_base[safe_columns].copy()
loan_base = loan_base.rename(columns={"Amount": "current_amount", "Value": "current_value"})

display(loan_base.head())
display(pd.DataFrame([{"table": "loan_base", "rows": len(loan_base), "columns": loan_base.shape[1]}]))


,TransactionId,CustomerId,BatchId,SubscriptionId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,current_amount,current_value,IssuedDateLoan,IsDefaulted,LoanId,LoanApplicationId,PayBackId,InvestorId
0,TransactionId_1,CustomerId_27,BatchId_2269,SubscriptionId_1,UGX,256,ProviderId_1,ProductId_7,airtime,ChannelId_1,-2000.0,2000.0,2018-11-27 18:14:22,0,LoanId_562,LoanApplicationId_91,PayBackId_1623,InvestorId_2
1,TransactionId_1000,CustomerId_398,BatchId_2532,SubscriptionId_7,UGX,256,ProviderId_1,ProductId_4,data_bundles,ChannelId_1,-7000.0,7000.0,2019-03-21 07:36:16,0,LoanId_871,LoanApplicationId_802,PayBackId_2042,InvestorId_1
2,TransactionId_1001,CustomerId_326,BatchId_2410,SubscriptionId_7,UGX,256,ProviderId_1,ProductId_7,airtime,ChannelId_1,-2000.0,2000.0,2019-01-19 13:54:22,0,LoanId_1093,LoanApplicationId_137,PayBackId_981,InvestorId_1
3,TransactionId_1005,CustomerId_502,BatchId_575,SubscriptionId_5,UGX,256,ProviderId_1,ProductId_3,airtime,ChannelId_1,-2000.0,2000.0,2019-03-28 11:37:16,0,LoanId_596,LoanApplicationId_472,PayBackId_65,InvestorId_2
4,TransactionId_1007,CustomerId_390,BatchId_2405,SubscriptionId_7,UGX,256,ProviderId_1,ProductId_7,airtime,ChannelId_1,-1000.0,1000.0,2019-01-30 10:40:38,0,LoanId_325,LoanApplicationId_871,PayBackId_1819,InvestorId_1


,table,rows,columns
0,loan_base,1153,18


In [10]:
train_context = train[train["IsDefaulted"].isna()].copy()
transaction_pool_columns = [
    "TransactionId", "CustomerId", "TransactionStartTime", "Amount", "Value",
    "ProviderId", "ProductId", "ProductCategory", "ChannelId", "PricingStrategy",
]
transaction_pool_columns = [column for column in transaction_pool_columns if column in train.columns]

transaction_history = pd.concat(
    [
        train_context[transaction_pool_columns],
        unlinked[[column for column in transaction_pool_columns if column in unlinked.columns]],
    ],
    ignore_index=True,
)
transaction_history = transaction_history.drop_duplicates()
transaction_history["txn_amount_abs"] = transaction_history["Amount"].abs()
transaction_history["txn_value_abs"] = transaction_history["Value"].abs()

display(transaction_history.head())
display(pd.DataFrame([{
    "table": "transaction_history",
    "rows": len(transaction_history),
    "columns": transaction_history.shape[1],
}]))


,TransactionId,CustomerId,TransactionStartTime,Amount,Value,ProviderId,ProductId,ProductCategory,ChannelId,txn_amount_abs,txn_value_abs
0,TransactionId_1683,CustomerId_27,2018-09-21 12:17:39,-550.0,550.0,ProviderId_1,ProductId_7,airtime,ChannelId_1,550.0,550.0
1,TransactionId_2235,CustomerId_27,2018-09-25 09:20:29,-550.0,550.0,ProviderId_1,ProductId_7,airtime,ChannelId_1,550.0,550.0
2,TransactionId_1053,CustomerId_27,2018-09-25 10:33:31,-550.0,550.0,ProviderId_1,ProductId_7,airtime,ChannelId_1,550.0,550.0
3,TransactionId_2633,CustomerId_27,2018-09-27 10:26:41,-1000.0,1000.0,ProviderId_1,ProductId_7,airtime,ChannelId_1,1000.0,1000.0
4,TransactionId_71,CustomerId_27,2018-09-27 12:44:21,-500.0,500.0,ProviderId_1,ProductId_7,airtime,ChannelId_1,500.0,500.0


,table,rows,columns
0,transaction_history,16939,11


## Prior transaction features
### 대출 전 거래 변수


In [11]:
loan_keys = loan_base[["TransactionId", "CustomerId", "IssuedDateLoan"]].rename(
    columns={"TransactionId": "LoanTransactionId"}
)

txn_pairs = loan_keys.merge(transaction_history, on="CustomerId", how="left")
txn_pairs = txn_pairs[
    txn_pairs["TransactionStartTime"].notna()
    & txn_pairs["IssuedDateLoan"].notna()
    & txn_pairs["TransactionStartTime"].lt(txn_pairs["IssuedDateLoan"])
].copy()
txn_pairs["days_since_txn"] = (txn_pairs["IssuedDateLoan"] - txn_pairs["TransactionStartTime"]).dt.days

grouped = txn_pairs.groupby("LoanTransactionId")
transaction_features = pd.DataFrame(index=loan_base["TransactionId"])
transaction_features.index.name = "TransactionId"
transaction_features["prior_txn_count_all"] = grouped.size()
transaction_features["prior_txn_active_days_all"] = grouped["TransactionStartTime"].apply(lambda values: values.dt.date.nunique())
transaction_features["prior_txn_recency_days_all"] = grouped["days_since_txn"].min()
transaction_features["prior_txn_amount_mean_all"] = grouped["txn_amount_abs"].mean()
transaction_features["prior_txn_amount_std_all"] = grouped["txn_amount_abs"].std()
transaction_features["prior_provider_nunique_all"] = grouped["ProviderId"].nunique()
transaction_features["prior_product_category_nunique_all"] = grouped["ProductCategory"].nunique()
transaction_features = transaction_features.fillna({
    "prior_txn_count_all": 0,
    "prior_txn_active_days_all": 0,
    "prior_txn_recency_days_all": 9999,
    "prior_txn_amount_mean_all": 0,
    "prior_txn_amount_std_all": 0,
    "prior_provider_nunique_all": 0,
    "prior_product_category_nunique_all": 0,
}).reset_index()

transaction_features["has_prior_txn"] = transaction_features["prior_txn_count_all"].gt(0).astype(int)
transaction_features["prior_txn_segment"] = pd.cut(
    transaction_features["prior_txn_count_all"],
    bins=[-1, 0, 2, 10, np.inf],
    labels=["no_prior_txn", "low_txn_history", "medium_txn_history", "high_txn_history"],
)

display(transaction_features.head())
display(pd.DataFrame([{
    "table": "transaction_features",
    "rows": len(transaction_features),
    "columns": transaction_features.shape[1],
}]))


,TransactionId,prior_txn_count_all,prior_txn_active_days_all,prior_txn_recency_days_all,prior_txn_amount_mean_all,prior_txn_amount_std_all,prior_provider_nunique_all,prior_product_category_nunique_all,has_prior_txn,prior_txn_segment
0,TransactionId_1,12.0,9.0,34.0,762.500000,307.774268,1.0,2.0,1,high_txn_history
1,TransactionId_1000,94.0,51.0,0.0,5546.808511,8283.442510,4.0,5.0,1,high_txn_history
2,TransactionId_1001,6.0,2.0,4.0,7361.666667,7317.377718,2.0,2.0,1,medium_txn_history
3,TransactionId_1005,2.0,1.0,1.0,50500.000000,70003.571337,2.0,2.0,1,low_txn_history
4,TransactionId_1007,10.0,7.0,5.0,5402.000000,4172.686931,3.0,3.0,1,medium_txn_history


,table,rows,columns
0,transaction_features,1153,10


## Prior repayment features
### 과거 상환 이력 변수


In [12]:
known_loans = loan_rows[[
    "TransactionId", "CustomerId", "IssuedDateLoan", "PaidOnDate", "DueDate", "IsDefaulted"
]].copy()
known_loans = known_loans.rename(columns={
    "TransactionId": "PriorLoanTransactionId",
    "IssuedDateLoan": "PriorIssuedDateLoan",
    "IsDefaulted": "prior_is_defaulted",
})
known_loans["known_outcome_date"] = np.where(
    known_loans["prior_is_defaulted"].eq(1),
    known_loans["DueDate"],
    known_loans["PaidOnDate"],
)
known_loans["known_outcome_date"] = pd.to_datetime(known_loans["known_outcome_date"], errors="coerce")

repay_pairs = loan_base[["TransactionId", "CustomerId", "IssuedDateLoan"]].rename(
    columns={"TransactionId": "CurrentTransactionId"}
).merge(known_loans, on="CustomerId", how="left")
repay_pairs = repay_pairs[
    repay_pairs["known_outcome_date"].notna()
    & repay_pairs["known_outcome_date"].lt(repay_pairs["IssuedDateLoan"])
].copy()

repay_grouped = repay_pairs.groupby("CurrentTransactionId")
repayment_features = pd.DataFrame(index=loan_base["TransactionId"])
repayment_features.index.name = "TransactionId"
repayment_features["prior_repayment_count"] = repay_grouped.size()
repayment_features["prior_default_count"] = repay_grouped["prior_is_defaulted"].sum()
repayment_features["prior_default_rate"] = repayment_features["prior_default_count"] / repayment_features["prior_repayment_count"]
repayment_features = repayment_features.fillna({
    "prior_repayment_count": 0,
    "prior_default_count": 0,
    "prior_default_rate": 0,
}).reset_index()
repayment_features["repayment_history_segment"] = np.where(
    repayment_features["prior_repayment_count"].eq(0),
    "cold_start_no_known_repayment",
    "has_prior_repayment_history",
)

display(repayment_features.head())
display(pd.DataFrame([{
    "table": "repayment_features",
    "rows": len(repayment_features),
    "columns": repayment_features.shape[1],
}]))


,TransactionId,prior_repayment_count,prior_default_count,prior_default_rate,repayment_history_segment
0,TransactionId_1,4.0,0.0,0.0,has_prior_repayment_history
1,TransactionId_1000,19.0,0.0,0.0,has_prior_repayment_history
2,TransactionId_1001,11.0,0.0,0.0,has_prior_repayment_history
3,TransactionId_1005,3.0,0.0,0.0,has_prior_repayment_history
4,TransactionId_1007,54.0,0.0,0.0,has_prior_repayment_history


,table,rows,columns
0,repayment_features,1153,5


## Final model table
### 최종 모델 테이블


In [13]:
model_table = (
    loan_base
    .merge(transaction_features, on="TransactionId", how="left")
    .merge(repayment_features, on="TransactionId", how="left")
)
model_table["IssuedDateLoan"] = pd.to_datetime(model_table["IssuedDateLoan"], errors="coerce")
model_table["issue_month"] = model_table["IssuedDateLoan"].dt.month
model_table["issue_weekday"] = model_table["IssuedDateLoan"].dt.day_name()
model_table["current_value_log"] = np.log1p(model_table["current_value"].abs())

output_path = PROCESSED_DIR / "train_model_table_v1.csv"
model_table.to_csv(output_path, index=False)

feature_summary = pd.DataFrame({
    "column": model_table.columns,
    "dtype": [str(model_table[column].dtype) for column in model_table.columns],
    "missing_rate_pct": [model_table[column].isna().mean() * 100 for column in model_table.columns],
})
feature_summary.to_csv(PROCESSED_DIR / "model_table_feature_summary.csv", index=False)

display(model_table.head())
display(pd.DataFrame([{
    "output": "train_model_table_v1.csv",
    "rows": len(model_table),
    "columns": model_table.shape[1],
}]))


,TransactionId,CustomerId,BatchId,SubscriptionId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,...,prior_product_category_nunique_all,has_prior_txn,prior_txn_segment,prior_repayment_count,prior_default_count,prior_default_rate,repayment_history_segment,issue_month,issue_weekday,current_value_log
0,TransactionId_1,CustomerId_27,BatchId_2269,SubscriptionId_1,UGX,256,ProviderId_1,ProductId_7,airtime,ChannelId_1,...,2.0,1,high_txn_history,4.0,0.0,0.0,has_prior_repayment_history,11,Tuesday,7.601402
1,TransactionId_1000,CustomerId_398,BatchId_2532,SubscriptionId_7,UGX,256,ProviderId_1,ProductId_4,data_bundles,ChannelId_1,...,5.0,1,high_txn_history,19.0,0.0,0.0,has_prior_repayment_history,3,Thursday,8.853808
2,TransactionId_1001,CustomerId_326,BatchId_2410,SubscriptionId_7,UGX,256,ProviderId_1,ProductId_7,airtime,ChannelId_1,...,2.0,1,medium_txn_history,11.0,0.0,0.0,has_prior_repayment_history,1,Saturday,7.601402
3,TransactionId_1005,CustomerId_502,BatchId_575,SubscriptionId_5,UGX,256,ProviderId_1,ProductId_3,airtime,ChannelId_1,...,2.0,1,low_txn_history,3.0,0.0,0.0,has_prior_repayment_history,3,Thursday,7.601402
4,TransactionId_1007,CustomerId_390,BatchId_2405,SubscriptionId_7,UGX,256,ProviderId_1,ProductId_7,airtime,ChannelId_1,...,3.0,1,medium_txn_history,54.0,0.0,0.0,has_prior_repayment_history,1,Wednesday,6.908755


,output,rows,columns
0,train_model_table_v1.csv,1153,34
